In [1]:
import numpy as np
import torch
import torch.nn.functional as F

import pandas as pd
from 意图识别训练.npl_model import Network_RNN

In [2]:
vocab = pd.read_csv("../datas/text_classify/vocab.csv", header=None, sep="\t")
vocab = np.array(vocab)
vocab_dict = dict(vocab)
vocab_dict.get('小品',1)

4174

In [3]:
corpus_id = pd.read_csv("../datas/text_classify/corpus_id.csv", header=None, sep="\t")
corpus_id

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,9315,2776,2391,6689,7440,6462,2983,89,7440,0,...,0,0,0,0,0,0,0,0,0,0
1,1634,9342,4851,3286,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9946,5378,805,10231,1130,9796,9588,7440,1801,9588,...,0,0,0,0,0,0,0,0,0,0
3,8085,7516,806,3432,7124,1279,7124,3260,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5011,4946,7516,5248,1170,5127,775,5072,7775,3313,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12095,829,2463,979,7901,2217,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12096,7748,4163,6887,4052,7763,10182,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12097,10454,9910,9492,7366,9896,5449,6062,10274,7898,2393,...,0,0,0,0,0,0,0,0,0,0
12098,7416,7465,8200,5683,6790,5868,8950,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
classes = ['Alarm-Update', 'Audio-Play', 'Calendar-Query', 'FilmTele-Play',
 'HomeAppliance-Control', 'Music-Play', 'Other', 'Radio-Listen',
 'TVProgram-Play', 'Travel-Query', 'Video-Play', 'Weather-Query']
classes = np.array(classes)
len(classes)

12

In [5]:
vocab_size = len(vocab)
num_classes = len(classes)
embed_dim = 128
hidden_size = 256
batch_size = 64
learning_rate = 0.01
epochs = 40
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [6]:
model = Network_RNN(vocab_size, embed_dim, hidden_size, num_classes)

best_model_path = "../datas/text_classify/model/RNN_last_model.pkl"
device = torch.device("mps" if torch.mps.is_available() else "cpu")

checkpoint = torch.load(best_model_path,map_location=device,weights_only=False)

best_acc = checkpoint['best_acc']

model.load_state_dict(checkpoint['model'])

model.to(device)
print(f"成功恢复，历史最佳准确率: {best_acc:.2f}%\n")




成功恢复，历史最佳准确率: 83.72%



In [7]:
import jieba
texts = ['循环播放赵本山的小品相亲来听','给我打开玩具房的灯','回放CCTV2的消费主张']
texts_cut = [jieba.lcut(text) for text in texts]
true_len = [len(text) for text in texts_cut]
print(texts_cut)
print(true_len)

/opt/anaconda3/envs/py3.12.11/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/71/13txvqn51x38p9qbcjvs2h940000gn/T/jieba.cache
Loading model cost 0.407 seconds.
Prefix dict has been built successfully.


[['循环', '播放', '赵本山', '的', '小品', '相亲', '来', '听'], ['给', '我', '打开', '玩具', '房', '的', '灯'], ['回放', 'CCTV2', '的', '消费', '主张']]
[8, 7, 5]


In [8]:
def token2id(text,vocab):
    ids = [vocab.get(char,1) for char in text]
    return ids


In [9]:
text_token = [token2id(text,vocab_dict) for text in texts_cut]
text_token

[[4776, 5378, 9111, 7440, 4174, 7494, 6035, 2992],
 [8085, 5011, 5082, 7192, 5040, 7440, 6857],
 [3293, 524, 7440, 6668, 1]]

In [10]:
text_maxlen = 30
from copy import deepcopy
text_token_padded = deepcopy(text_token)
for i in text_token_padded:
    if len(i)<text_maxlen:
        i.extend([0] * (text_maxlen-len(i)))
    elif len(i)>text_maxlen:
        i = i[:text_maxlen]


In [11]:
print(text_token_padded[:10])
print(max([len(i) for i in text_token_padded]))
print(min([len(i) for i in text_token_padded]))

[[4776, 5378, 9111, 7440, 4174, 7494, 6035, 2992, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8085, 5011, 5082, 7192, 5040, 7440, 6857, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3293, 524, 7440, 6668, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
30
30


In [12]:
data = torch.tensor(text_token_padded)
true_len = torch.tensor(true_len)
print(data)
print(true_len)

tensor([[4776, 5378, 9111, 7440, 4174, 7494, 6035, 2992,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0],
        [8085, 5011, 5082, 7192, 5040, 7440, 6857,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0],
        [3293,  524, 7440, 6668,    1,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0]])
tensor([8, 7, 5])


In [46]:
k = 3
model.eval()

with torch.no_grad():

    data, true_len = data.to(device), true_len.cpu()
    output = model(data, true_len)
    probs = F.softmax(output, dim=-1)
    probs = probs.argsort(dim=1,descending=True)[:,:k]
    print(probs)




    pred = output.argmax(dim=1, keepdim=True)
    print(pred)

    print(classes[pred.cpu().numpy().astype(int)])
    print(classes[probs.cpu().numpy().astype(int)])


tensor([[ 1,  5, 10],
        [ 4,  0,  2],
        [10,  9,  6]], device='mps:0')
tensor([[ 1],
        [ 4],
        [10]], device='mps:0')
[['Audio-Play']
 ['HomeAppliance-Control']
 ['Video-Play']]
[['Audio-Play' 'Music-Play' 'Video-Play']
 ['HomeAppliance-Control' 'Alarm-Update' 'Calendar-Query']
 ['Video-Play' 'Travel-Query' 'Other']]
